# 03 — Refusal signal calibration

Is `margin` actually correlated with regex-refusal? Choose a threshold.


In [ ]:
from safety_circuits.config import MODELS, REPO_ROOT
from safety_circuits.models import load_model
from safety_circuits.refusal import score_refusal
from safety_circuits.data import load_jsonl
import pandas as pd


In [ ]:
loaded = load_model(MODELS['tinyllama'])
pairs = load_jsonl(REPO_ROOT / 'data' / 'processed' / 'pairs.jsonl')[:50]


In [ ]:
rows = []
for r in pairs:
    for kind in ('harm', 'safe'):
        s = score_refusal(loaded, r[kind]['text'])
        rows.append({'kind': kind, 'margin': s.margin, 'p_ref': s.p_refusal,
                     'regex': int(s.refused_regex)})
df = pd.DataFrame(rows)
df.groupby('kind').agg(['mean', 'std'])


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
for kind, g in df.groupby('kind'):
    ax.hist(g['margin'], bins=20, alpha=0.6, label=kind)
ax.set_xlabel('refusal margin (log-prob)')
ax.legend(); ax.set_title('margin separates harm vs safe?')


Pick the threshold where the two histograms cross — that's your decision boundary.
